# 두 단계 탐지: 남은 학습을 한 번에 실행

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/capstone-privai/promptguard-model/blob/main/notebooks/colab_two_stage.ipynb)

CredData-trajectory v1로 아래 단계를 차례로 학습합니다. 설계는 `docs/ml_span/TOKEN_SPAN_SIZE_SERIES.md`와 `docs/ml_span/VERIFIER_PLAN.md`에 있습니다.

| 단계 | 내용 | 필요한 GPU |
|---|---|---|
| `verifier` | 탐지기 후보 만들기와 오탐 분석, 뒷단 검증기(mmBERT-small) 학습, validation 정책 비교 | T4 |
| `mmbert-base` | span 모델 mmBERT-base | T4 |
| `qwen1.5b-bidir` | span 모델 양방향 Qwen 1.5B | T4 |
| `qwen7b-bidir` | span 모델 양방향 Qwen 7B | L4 또는 A100 |
| `qwen14b-bidir` | span 모델 양방향 Qwen 14B | A100 40GB 이상 |

1. `CredData-trajectory.zip`이 Drive의 `promptguard-runs/`에 있어야 합니다(#20에서 쓴 것과 같은 파일).
2. **런타임 → 런타임 유형 변경 → GPU**를 고른 뒤 **런타임 → 모두 실행**을 누릅니다. Drive 연결을 허용합니다.
3. 이 GPU로 못 하는 단계는 건너뛰고 알려 줍니다. 예: T4에서는 7B와 14B를 건너뜁니다.
4. 끝난 단계는 다시 실행해도 건너뜁니다. Qwen은 중간 체크포인트에서 이어서 학습합니다. 런타임이 끊기면 **모두 실행**을 다시 누르면 됩니다.
   - 권장 순서: 먼저 T4로 모두 실행해 위 세 단계를 끝냅니다. 그다음 A100으로 바꿔 다시 모두 실행하면 7B와 14B만 돌아갑니다.

**지켜야 할 규칙**
- epoch과 cutoff는 validation에서만 고릅니다. test는 `RUN_TEST = True`로 바꾼 마지막 셀에서, 모든 모델을 고정한 뒤 한 번만 채점합니다.
- 결과에는 개수만 있습니다. 가중치와 후보 데이터는 Drive에만 두고 Git에 올리지 않습니다.

In [ ]:
# ── 설정: 여기만 고치세요 ──
STEPS = ['verifier', 'mmbert-base', 'qwen1.5b-bidir', 'qwen7b-bidir', 'qwen14b-bidir']  # 위에서부터 차례로 실행
RUN_TEST = False   # True: 모든 모델을 고정한 뒤 test를 한 번만 채점합니다(이미 채점한 모델은 건너뜀)
REPO_REF = 'main'  # 학습에 쓸 코드의 브랜치 또는 태그
DRIVE_DIR = '/content/drive/MyDrive/promptguard-runs'  # 데이터 zip과 결과가 있는 곳(세션이 끊겨도 남습니다)

In [ ]:
# GPU 확인: 없으면 멈추고, 이 GPU로 할 수 없는 단계는 건너뜁니다.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. 런타임 → 런타임 유형 변경 → GPU를 고른 뒤 다시 실행하세요.')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
GPU_MAJOR = torch.cuda.get_device_capability(0)[0]
# (필요한 GPU 메모리 GB, compute capability): 7B와 14B는 bfloat16이라 Ampere(8) 이상이어야 합니다.
NEEDS = {'qwen7b-bidir': (22, 8), 'qwen14b-bidir': (38, 8)}
KNOWN = {'verifier', 'mmbert-base', 'qwen1.5b-bidir', 'qwen7b-bidir', 'qwen14b-bidir'}
unknown = [s for s in STEPS if s not in KNOWN]
if unknown:
    raise ValueError(f'모르는 단계: {unknown}')
RUNNABLE = [s for s in STEPS if s not in NEEDS or (GPU_GB >= NEEDS[s][0] and GPU_MAJOR >= NEEDS[s][1])]
print(GPU_NAME, round(GPU_GB, 1), 'GB')
print('실행할 단계:', RUNNABLE)
print('이 GPU로는 건너뜀:', [s for s in STEPS if s not in RUNNABLE])

In [ ]:
# Google Drive 연결과 데이터 zip 확인
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
if not any(name.startswith('CredData-trajectory') and name.endswith('.zip') for name in os.listdir(DRIVE_DIR)):
    print('주의: promptguard-runs/에 CredData-trajectory.zip이 없습니다. 준비 스크립트가 My Drive 전체에서 찾아봅니다.')

In [ ]:
# 저장소 받기와 설치
import subprocess

def sh(cmd):
    """Run a shell command and stream its output into the notebook."""
    print('$', cmd, flush=True)
    p = subprocess.Popen(['bash', '-o', 'pipefail', '-c', cmd], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        raise RuntimeError(f'명령 실패 (exit {p.returncode}): {cmd}')

REPO_DIR = '/content/promptguard-model'
if not os.path.exists(REPO_DIR):
    sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {REPO_REF} https://github.com/capstone-privai/promptguard-model {REPO_DIR}')
else:  # 같은 세션에서 다시 실행할 때 옛 코드로 학습하지 않도록 최신 커밋으로 바꿉니다.
    sh(f'cd {REPO_DIR} && GIT_LFS_SKIP_SMUDGE=1 git fetch -q --depth 1 origin {REPO_REF} && git checkout -q FETCH_HEAD')
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('commit', COMMIT)
# 로컬과 같은 transformers 버전. torch는 Colab에 설치된 CUDA 버전을 그대로 씁니다.
sh('pip install -q transformers==4.57.6 "peft>=0.13,<1" && pip install -q -e .')
os.environ.update(DRIVE_DIR=DRIVE_DIR, PYTHONPATH='src', USE_TF='0')

In [ ]:
# gitleaks: 배포 탐지기의 투표 모듈 중 하나입니다. 받지 못하면 그 모듈만 빠지고, 빠진 사실이 후보 manifest에 남습니다.
GITLEAKS = '8.18.4'
try:
    sh(f'command -v gitleaks || (curl -sSL https://github.com/gitleaks/gitleaks/releases/download/v{GITLEAKS}/'
       f'gitleaks_{GITLEAKS}_linux_x64.tar.gz | tar -xz -C /usr/local/bin gitleaks) && gitleaks version')
except RuntimeError as error:
    print('gitleaks 없이 진행합니다:', error)

In [ ]:
# 학습: 끝난 단계는 건너뜁니다. 한 단계가 실패해도 다음 단계로 넘어가고, 실패는 마지막에 모아 보여 줍니다.
def done(step):
    if step == 'verifier':
        return os.path.exists(f'{DRIVE_DIR}/verifier-creddata-mmbert-small/system_validation.json')
    return os.path.exists(f'{DRIVE_DIR}/token-span-creddata-{step}/selection.json')

FAILED = {}
for step in RUNNABLE:
    if done(step):
        print(f'== {step}: 이미 끝남, 건너뜀')
        continue
    print(f'== {step}', flush=True)
    try:
        if step == 'verifier':
            sh('bash scripts/colab_verifier.sh prepare')
            if not os.path.exists(f'{DRIVE_DIR}/verifier-creddata-mmbert-small/model'):
                sh('bash scripts/colab_verifier.sh train mmbert-small')
            sh('bash scripts/colab_verifier.sh system mmbert-small')
        else:
            sh(f'bash scripts/colab_token_span.sh {step}')
    except RuntimeError as error:
        FAILED[step] = str(error)
        print(f'!! {step} 실패: {error}', flush=True)
print('실패한 단계:', FAILED or '없음')

In [ ]:
# validation 결과 요약(개수만). test는 열지 않습니다.
import json

def load(path):
    with open(path) as f:
        return json.load(f)

for step in STEPS:
    if step == 'verifier':
        root = f'{DRIVE_DIR}/verifier-creddata-mmbert-small'
        if os.path.exists(f'{DRIVE_DIR}/data-verifier-creddata-traj-v1/analysis.json'):
            a = load(f'{DRIVE_DIR}/data-verifier-creddata-traj-v1/analysis.json')['validation']
            print('== 후보 (validation): 정답', a['gold_spans'], '| 후보', a['candidates'], '| strength별', json.dumps(a['by_tier'], ensure_ascii=False))
            print('   정답에 닿은 후보:', json.dumps(a['gold_reached'], ensure_ascii=False), '| 어떤 후보도 닿지 않은 정답', a['gold_missed_by_every_candidate'])
        if os.path.exists(f'{root}/system_validation.json'):
            s = load(f'{root}/system_validation.json')
            print('== 검증기 정책 비교 (validation, 기준', round(s['threshold'], 4), ')')
            for policy in ('union', 'strong_only', 'strong_verifier'):
                print(f'   {policy:16s}', {k: s[policy][k] for k in ('masked', 'false_masks', 'gold_reached', 'precision', 'recall', 'f2')})
        continue
    path = f'{DRIVE_DIR}/token-span-creddata-{step}/selection.json'
    if not os.path.exists(path):
        print(f'== {step}: 아직 없음')
        continue
    s = load(path)
    best = next(h for h in s['history'] if h['epoch'] == s['selected']['epoch'])['selected']
    print(f"== {step}: epoch {s['selected']['epoch']}, cutoff {best['threshold']:.3f}, "
          f"TP/FP/FN {best['tp']}/{best['fp']}/{best['fn']}, precision {best['precision'] or 0:.3f}, F2 {best['f2'] or 0:.3f}")

In [ ]:
# test 채점: RUN_TEST = True일 때만, 모델마다 한 번만 합니다.
if RUN_TEST:
    spans = [s for s in STEPS if s != 'verifier' and os.path.exists(f'{DRIVE_DIR}/token-span-creddata-{s}/selection.json')
             and not os.path.exists(f'{DRIVE_DIR}/token-span-creddata-{s}/test.json')]
    if spans:
        sh('bash scripts/colab_token_span.sh test ' + ' '.join(spans))
    if 'verifier' in STEPS and os.path.exists(f'{DRIVE_DIR}/verifier-creddata-mmbert-small/model') \
            and not os.path.exists(f'{DRIVE_DIR}/verifier-creddata-mmbert-small/final'):
        sh('bash scripts/colab_verifier.sh test mmbert-small')
else:
    print('RUN_TEST = False: test는 채점하지 않았습니다.')